# Part 2: K-Means Clustering — Finding Natural Segments
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Wednesday — Customer Segmentation

Yesterday Sarah looked at the data in 2D. Today she finds the actual groups. She wants 3–6 customer segments — enough to be meaningful, few enough to actually act on with NorthStar's marketing team.

K-Means is the standard tool. Three big questions to answer:
1. **What K?** (How many clusters?)
2. **How well do they actually cluster?** (Silhouette score)
3. **What does each cluster MEAN?** (Profiling — naming each one)

**By the end of this notebook you will be able to:**
- Fit K-Means in one line
- Choose K with the elbow method + silhouette score
- Profile each cluster (what makes it distinctive)
- Name your clusters for a non-technical stakeholder

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — KMeans ready")

## Step 1 — Setup (same preprocessing as NB 02)

In [ ]:
df = pd.read_csv("data/northstar_customers.csv")
features = df.drop(columns=["customer_id", "churned"])

numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

X_processed = preprocessor.fit_transform(features)
print(f"Processed shape: {X_processed.shape}")

## Step 2 — How to pick K?

K-Means doesn't tell you K. You have to decide. Two standard heuristics:

1. **Elbow method** — plot inertia (within-cluster sum of squares) vs K. Look for the bend.
2. **Silhouette score** — how well each point fits its cluster vs the next-best one. Higher = better.

We'll compute both for K = 2, 3, ..., 10.

In [ ]:
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_processed)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_processed, labels))

# Plot both
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(K_range, inertias, "o-", linewidth=2, color="steelblue", markersize=10)
axes[0].set_xlabel("K (number of clusters)")
axes[0].set_ylabel("Inertia (within-cluster sum of squares)")
axes[0].set_title("Elbow method — look for the bend")

axes[1].plot(K_range, silhouettes, "s-", linewidth=2, color="coral", markersize=10)
axes[1].set_xlabel("K (number of clusters)")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette score — higher is better")

plt.tight_layout()
plt.show()

print("K     Inertia    Silhouette")
for k, inertia, sil in zip(K_range, inertias, silhouettes):
    print(f"  {k}   {inertia:10.0f}     {sil:.4f}")

## ⏸️ Pause and Predict

Look at the elbow plot and silhouette plot. Before reading on, decide:
- What K would you pick?
- What would you tell Marcus if he asked WHY?

*Your prediction:*

> *Hints to consider:*
> - The elbow plot's "bend" is rarely sharp on real data. Often it's a smooth curve.
> - Silhouette score peaks at SOME K — that's a vote for that K.
> - **Business judgement matters.** K=2 might score best silhouette but 2 segments isn't enough to act on. K=8 might be slightly worse silhouette but harder to brief to a marketing team. Sweet spot: 3-5 for most business contexts.

## Step 3 — Pick K = 4

We'll commit to K = 4 — a common segmentation count for marketing teams. (Silhouette score probably suggests K=2 or K=3 for purely "tightest" clustering, but 4 segments is more actionable for NorthStar.)

In [ ]:
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = km.fit_predict(X_processed)

# Add cluster labels to the original dataframe (with raw features)
df_clustered = features.copy()
df_clustered["cluster"] = labels

# Cluster sizes
print(f"K = {K}")
print()
print("Cluster sizes:")
print(df_clustered["cluster"].value_counts().sort_index().to_string())

## Step 4 — Visualise clusters in PCA space

Project to 2D with PCA (from NB 02) and colour by cluster. This is the picture Sarah will put on the Friday slides.

In [ ]:
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_processed)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#1F77B4", "#FF7F0E", "#2CA02C", "#D62728"]
for c in range(K):
    mask = labels == c
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                alpha=0.5, s=12, color=colors[c],
                label=f"Cluster {c} ({mask.sum()} customers)")
ax.set_xlabel(f"PC1  ({pca_2d.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2  ({pca_2d.explained_variance_ratio_[1]:.1%} var)")
ax.set_title(f"NorthStar customers — K-Means with K = {K}")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

### 💡 What you should notice

- The clusters are **visible in PCA space**, even though the 2D coloured cloud wasn't obviously separable in NB 02.
- K-Means is using ALL 17 dimensions to find the groups — the 2D plot is just a projection of that result.
- Some overlap is normal. K-Means assigns every customer to exactly ONE cluster (hard assignment) — there are no "in-between" customers.

## Step 5 — Cluster profiles: what makes each cluster distinctive?

For each cluster, compute the mean of every feature. Compare to the global mean to spot what each cluster is about.

In [ ]:
# Profile each cluster on the ORIGINAL (unscaled) numeric features
numeric_cols = ["age", "tenure_months", "num_purchases_quarter",
                "avg_monthly_spend_gbp", "returns_per_purchase",
                "last_login_days_ago", "avg_review_polarity",
                "support_tickets_quarter"]

profile = df_clustered.groupby("cluster")[numeric_cols].mean().round(2)
global_mean = features[numeric_cols].mean().round(2)
profile.loc["GLOBAL_MEAN"] = global_mean

print("Cluster profiles vs global mean:")
print(profile.to_string())

## Step 6 — Heatmap of standardised differences

A heatmap makes the "what's different" pattern visible at a glance. Standardise each cluster's mean by subtracting the global mean and dividing by the global standard deviation.

In [ ]:
# Z-score each cluster's mean against the global distribution
global_std  = features[numeric_cols].std()
cluster_means = df_clustered.groupby("cluster")[numeric_cols].mean()
z_scores = (cluster_means - global_mean) / global_std

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(z_scores, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1.5, vmax=1.5, cbar_kws={"label": "Z-score vs global mean"}, ax=ax)
ax.set_title(f"Cluster profiles — features as Z-scores against the global distribution\n(red = above average, blue = below average)")
plt.tight_layout()
plt.show()

### 💡 What this tells us — and how to name the clusters

Look at each row of the heatmap and write one sentence:

- **Cluster 0** — high on (X, Y, Z); low on (A, B). Call it: **"name"**
- **Cluster 1** — high on (X, Y, Z); low on (A, B). Call it: **"name"**
- **Cluster 2** — high on (X, Y, Z); low on (A, B). Call it: **"name"**
- **Cluster 3** — high on (X, Y, Z); low on (A, B). Call it: **"name"**

Common patterns for retail customer data:
- *Loyal high-value* — long tenure, high spend, low returns, satisfied (positive polarity)
- *New + frustrated* — short tenure, low spend, high returns, lots of support tickets
- *Steady mid-tier* — close to global mean on everything (the "average customer")
- *Bargain hunters / churn risk* — short tenure, low spend, high returns

Sarah's task on Friday is to name them. We've given her the data.

## ✅ Section Summary

| What we did | Output |
|---|---|
| **Computed elbow + silhouette** for K = 2-10 | Two metrics for choosing K |
| **Picked K = 4** | Business judgement — actionable number of segments |
| **Fit K-Means** | 4 cluster labels, one per customer |
| **Plotted in PCA space** | Visible groupings even when raw 2D wasn't obvious |
| **Profiled each cluster** | Per-cluster feature means + heatmap of Z-score differences |
| **Named the segments** | The output Marcus actually wants — meaningful labels, not numbers |

**Key insight for our scenario:**
> K-Means alone gives Sarah numbers (cluster 0, 1, 2, 3). The VALUE comes from profiling — translating "cluster 2" into "Loyal high-value customers, 26% of base, candidates for the premium upsell campaign."

---
**Up next → Part 3:** Thursday — Isolation Forest for anomaly detection. Find unusual customers worth investigating.
Open `04_isolation_forest.ipynb`

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## Extension 1 — Reproducibility: K-Means and random_state

K-Means starts from random initial centroids. With `n_init=10` (default), it tries 10 different starting points and keeps the best. But two runs with different `random_state` can still give slightly different cluster ASSIGNMENTS — even if the clusters themselves are very similar.

ALWAYS set `random_state` for reproducible work.

In [ ]:
# Compare K-Means with two different random_states
km_42  = KMeans(n_clusters=4, random_state=42, n_init=10)
km_123 = KMeans(n_clusters=4, random_state=123, n_init=10)

labels_42  = km_42.fit_predict(X_processed)
labels_123 = km_123.fit_predict(X_processed)

# Cluster sizes
sizes_42  = pd.Series(labels_42).value_counts().sort_index().values
sizes_123 = pd.Series(labels_123).value_counts().sort_index().values

print(f"Cluster sizes with random_state=42:   {sizes_42}")
print(f"Cluster sizes with random_state=123:  {sizes_123}")

# Pairwise agreement — what fraction of customers got the SAME cluster?
# (Note: cluster IDs might be permuted between the two runs. Best to use ARI.)
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(labels_42, labels_123)
print()
print(f"Adjusted Rand Index between the two runs: {ari:.3f}")
print("→ ARI ≈ 1 means same clustering. ARI ≈ 0 means random agreement.")
print("→ K-Means is usually stable on tabular data; random_state matters mostly for getting EXACT reproducibility.")

## Extension 2 — When K-Means struggles: non-globular clusters

K-Means assumes clusters are SPHERICAL (roughly equal in all directions). When the natural clusters are elongated, stringy, or irregularly shaped, K-Means fails — it carves up the natural shapes into pieces.

For non-globular structure, use **DBSCAN** (density-based) — covered in `optional_extensions.ipynb`.

For now, here's a synthetic demo to make the limitation visible.

In [ ]:
from sklearn.datasets import make_moons

# Two interlocking moons — clearly two groups, but not spherical
X_moons, _ = make_moons(n_samples=400, noise=0.05, random_state=42)

km_demo = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_demo = km_demo.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], s=20, color="gray", alpha=0.6)
axes[0].set_title("Two natural 'moons' — humans see clear groups")

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_demo, cmap="coolwarm", s=20, alpha=0.7)
axes[1].set_title("K-Means with K=2 → carves the moons in half (the wrong way)")

plt.tight_layout()
plt.show()
print("→ K-Means fails on non-globular shapes. Use DBSCAN if you suspect this.")

## Extension 3 — Mini-batch K-Means for large datasets

For very large datasets (>100k rows), regular K-Means is slow. `MiniBatchKMeans` uses random subsets each iteration — much faster, almost as accurate.

In [ ]:
from sklearn.cluster import MiniBatchKMeans
import time

# Time regular vs mini-batch
start = time.time()
km_full = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_processed)
t_full = time.time() - start

start = time.time()
km_mini = MiniBatchKMeans(n_clusters=4, random_state=42, batch_size=512, n_init=10).fit(X_processed)
t_mini = time.time() - start

# Compare cluster qualities
sil_full = silhouette_score(X_processed, km_full.labels_)
sil_mini = silhouette_score(X_processed, km_mini.labels_)

print(f"Full KMeans:    time = {t_full*1000:.0f}ms, silhouette = {sil_full:.4f}")
print(f"MiniBatch:      time = {t_mini*1000:.0f}ms, silhouette = {sil_mini:.4f}")
print()
print("MiniBatchKMeans is the right choice when full KMeans takes >10s.")
print("For our 10k-row dataset, the difference is small. For 1M+ rows it's enormous.")